# 06: Range Trees

*Authors: Felix Espey, Kevin Buchin*

This notebook serves as supplementary learning material for the course **Geometric Algorithms**.
It showcases and explains implementations of algorithms presented in the corresponding lecture, and elaborates on some practical considerations concerning their use.
Furthermore, it offers interactive visualisations and animations.

## Table of Contents

1. Introduction
2. Range Searching in one Dimension
3. Range Searching in two Dimensions (Kd-Trees)
4. Range Trees
5. References

## 1. Introduction



Change these parameters to whatever boundaries you want to use

In [1]:
LOWER_BOUND = 80
UPPER_BOUND = 200

find first node that is between lowerbound and upper obund.

color: blue = path through tree

red: leaf found that is outside of range -> equals a return value of None

green: node in range found

In [2]:
from modules.data_structures.binary_trees import BST, EST, TreeTracker, TransitionType, TransitionEvent
from modules.geometry import IntComparator, Point
import random
from modules.visualisation import VisualisationTool, BinaryTreeInstance, BinaryTreeMode
from notebooks.modules import AnimationObject

from modules import Node, IntComparator, TreeTracker, ComparisonResult, RangeSearchAnimator, RangeSearchMode
from typing import Optional

int_comparator = IntComparator()

def find_splitting_node(node : Node[int, None], lower_bound : int, upper_bound : int, rta : RangeSearchAnimator) -> Node[int, None] | None:
    cr_left = int_comparator.compare(lower_bound, node.key)
    cr_right = int_comparator.compare(upper_bound, node.key)
    if cr_right is ComparisonResult.BEFORE:
        #range fully left of node
        if node.left is None:
            rta.tag_cur_node(3)
            return None
        else:
            rta.tag_cur_node(1)
            rta.go_to_left_child()
            return find_splitting_node(node.left, lower_bound, upper_bound, rta)
    elif cr_left is ComparisonResult.AFTER:
        #range fully right of node
        if node.right is None:
            rta.tag_cur_node(3)
            return None
        else:
            rta.tag_cur_node(1)
            rta.go_to_right_child()
            return find_splitting_node(node.right, lower_bound, upper_bound, rta)
    else:
        #node in range
        rta.tag_cur_node(2)
        rta.save_node() # needed to find node in range search
        return node

def run_find_splitting_node(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    if LOWER_BOUND < UPPER_BOUND:
       find_splitting_node(est._root, LOWER_BOUND, UPPER_BOUND, rta)
    return rta

bti = BinaryTreeInstance()
vis = VisualisationTool(400,400, bti)
vis.register_algorithm("find splitting node", run_find_splitting_node, RangeSearchMode())
vis.display()

smaller than/greater than methods

blue: path through tree

red: leaves that where checked but are to big/small

green: leaves that where returned

In [3]:
def leaves(node : Node[int, None], rta:RangeSearchAnimator):
    if node.is_leaf():
        rta.tag_cur_node(2)
    else:
        rta.tag_cur_node(1)
        rta.go_to_left_child()
        leaves(node.left, rta)
        rta.go_to_parent()
        rta.go_to_right_child()
        leaves(node.right, rta)
        rta.go_to_parent()

def less_or_equal(node : Node[int, None], upper_bound : int, rta : RangeSearchAnimator):
    cr = int_comparator.compare(upper_bound, node.key)
    if cr is ComparisonResult.MATCH or cr is ComparisonResult.AFTER:
        #less than search term
        if not node.is_leaf():
            rta.go_to_left_child()
            leaves(node.left, rta)
            rta.go_to_parent()
            rta.tag_cur_node(1)
            rta.go_to_right_child()
            less_or_equal(node.right, upper_bound, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(2)
    else:
        #more than search term
        if not node.is_leaf():
            rta.tag_cur_node(1)
            rta.go_to_left_child()
            less_or_equal(node.left, upper_bound, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(3)

def greater_or_equal(node : Node[int, None], lower_bound : int, rta : RangeSearchAnimator):
    cr = int_comparator.compare(lower_bound, node.key)
    if cr is ComparisonResult.BEFORE or cr is ComparisonResult.MATCH:
        #more than search term
        if not node.is_leaf():
            rta.go_to_left_child()
            greater_or_equal(node.left, lower_bound, rta)
            rta.go_to_parent()
            rta.tag_cur_node(1)
            rta.go_to_right_child()
            leaves(node.right, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(2)
    else:
        #less than search term
        if not node.is_leaf():
            rta.tag_cur_node(1)
            rta.go_to_right_child()
            greater_or_equal(node.right, lower_bound, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(3)

def run_less_or_equal(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    less_or_equal(est._root, UPPER_BOUND, rta)
    return rta

def run_greater_or_equal(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    greater_or_equal(est._root, LOWER_BOUND, rta)
    return rta





vis.register_algorithm("report smaller than upper bound", run_less_or_equal, RangeSearchMode())
vis.register_algorithm("report bigger than lower bound", run_greater_or_equal, RangeSearchMode())
vis.display()

In [4]:
def range_search(node : Node[int, None], lower_bound: int, upper_bound : int, rta : RangeSearchAnimator) -> list[Node[int, None]]:
        splitting_node = find_splitting_node(node, lower_bound, upper_bound, rta)
        if splitting_node is None:
            return []
        if splitting_node.is_leaf():
            return [splitting_node]
        else:
            print(rta._cur_level)
            print(rta._cur_node)
            rta.load_node()
            rta.tag_cur_node(1)
            result = []
            if splitting_node.left is not None:
                rta.go_to_left_child()
                greater_or_equal(splitting_node.left, lower_bound, rta)
                rta.go_to_parent()
            if splitting_node.right is not None:
                rta.go_to_right_child()
                less_or_equal(splitting_node.right, upper_bound, rta)
                rta.go_to_parent()
            return result


def run_range_search(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    if LOWER_BOUND < UPPER_BOUND:
        range_search(est._root, LOWER_BOUND, UPPER_BOUND, rta)
    return rta

vis.register_algorithm("report in range", run_range_search, RangeSearchMode())
vis.display()
vis._random_button.click()
vis._algorithm_buttons[3].click()

0
0


## 2. Range Searching in one Dimension

TODO

## 3. Range Searching in two Dimensions (Kd-Trees)
TODO

## 4. Range Trees
TODO